# 01 — Exploratory Data Analysis (EDA)

This notebook uses the project’s SQLite feature store: `data/processed/churn_features.db`.

**Run order**
- Build DB + feature table
- Explore churn patterns
- Generate EDA plot pack to `assets/eda/`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
# Build the SQLite feature store (safe to re-run)
from src.data_processing import main as build_db

build_db()

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = PROJECT_ROOT / 'data' / 'processed' / 'churn_features.db'

with sqlite3.connect(DB_PATH) as conn:
    raw = pd.read_sql('SELECT * FROM raw_data', conn)
    feat = pd.read_sql('SELECT * FROM churn_features', conn)

raw.shape, feat.shape

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

churn_rate = raw['Churn'].mean() * 100
print(f'Overall churn rate: {churn_rate:.1f}%')

fig, ax = plt.subplots(figsize=(5.2, 4.2))
counts = raw['Churn'].value_counts().sort_index()
ax.pie(
    counts,
    labels=['No Churn', 'Churn'],
    autopct='%1.1f%%',
    startangle=140,
    colors=['#4C9BE8', '#E8694C'],
    explode=[0, 0.06],
)
ax.set_title('Overall Churn Distribution', fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Churn rate across key categoricals (same spirit as Kaggle notebook)
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = raw.groupby(col)['Churn'].mean().sort_values(ascending=False)
    ct.plot(kind='bar', ax=axes[i], edgecolor='black', width=0.65)
    axes[i].set_title(f'Churn Rate by {col}', fontsize=11)
    axes[i].set_ylabel('Churn Rate')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=25)
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.1%}',
            ha='center',
            va='bottom',
            fontsize=8,
        )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Churn Rate Across Categorical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Business-friendly views from SQL-derived features
order = ['0-1yr', '1-2yr', '2-4yr', '4+yr']

tmp = feat.groupby('tenure_bucket', as_index=False)['Churn'].mean()
tmp['churn_rate_pct'] = tmp['Churn'] * 100

fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.barplot(data=tmp, x='tenure_bucket', y='churn_rate_pct', order=order, ax=ax)
ax.set_ylabel('Churn rate (%)')
ax.set_xlabel('Tenure bucket')
ax.set_title('Churn Rate by Tenure Bucket', fontsize=13, fontweight='bold')
for p in ax.patches:
    ax.annotate(
        f"{p.get_height():.1f}%",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center',
        va='bottom',
    )
plt.show()

pivot = feat.pivot_table(index='TechSupport', columns='OnlineSecurity', values='Churn', aggfunc='mean')
fig, ax = plt.subplots(figsize=(6.8, 4.8))
sns.heatmap(pivot * 100, annot=True, fmt='.1f', cmap='Reds', linewidths=0.5, ax=ax)
ax.set_title('Churn Rate (%) — TechSupport × OnlineSecurity', fontsize=12.5, fontweight='bold')
plt.show()

In [ ]:
# Generate the full plot pack to assets/eda/
from src.eda_plots import generate_eda_plots

generate_eda_plots()

print('Saved plots to:', (PROJECT_ROOT / 'assets' / 'eda').resolve())